In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# timezone for display (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs
metric = pd.read_csv("metric.csv")
trace = pd.read_csv("trace.csv")
log = pd.read_csv("log.csv")
error_logs = pd.read_csv("error_logs.csv")

# Ensure value columns are float where applicable
for df, col in [(metric, 'value'), (trace, 'value'), (log, 'value')]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# 1) metric.csv aggregations (global)
metric_agg_full = (
    metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(count='count',
         mean='mean',
         min='min',
         max='max',
         p95=lambda x: x.quantile(0.95))
)

# Round numeric columns for compactness
metric_agg_full[['mean','min','max','p95']] = metric_agg_full[['mean','min','max','p95']].round(6)

metric_summary = metric_agg_full.sort_values('count', ascending=False).head(20)

# 2) trace.csv aggregations (global)
trace_agg_full = (
    trace
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .agg(count='count',
         mean='mean',
         min='min',
         max='max',
         p95=lambda x: x.quantile(0.95))
)
trace_agg_full[['mean','min','max','p95']] = trace_agg_full[['mean','min','max','p95']].round(6)

trace_summary = trace_agg_full.sort_values('count', ascending=False).head(20)

# 3) log.csv aggregations (global)
log_agg_full = (
    log
    .groupby(['cmdb_id', 'log_name'], as_index=False)['value']
    .agg(count='count',
         mean='mean',
         min='min',
         max='max',
         p95=lambda x: x.quantile(0.95))
)
log_agg_full[['mean','min','max','p95']] = log_agg_full[['mean','min','max','p95']].round(6)

log_summary = log_agg_full.sort_values('count', ascending=False).head(20)

# 4) error_logs.csv summary per cmdb_id
# Convert timestamp to timezone-aware datetimes (UTC -> convert to Asia/Shanghai)
if 'timestamp' in error_logs.columns:
    ts_dt = pd.to_datetime(error_logs['timestamp'], unit='s', utc=True)
    ts_dt = ts_dt.dt.tz_convert(tz)
    error_logs['_ts_dt'] = ts_dt
else:
    error_logs['_ts_dt'] = pd.NaT

error_agg_full = (
    error_logs
    .groupby('cmdb_id', as_index=False)
    .agg(count=('message','count'),
         earliest_ts=('_ts_dt','min'),
         latest_ts=('_ts_dt','max'))
)

# Keep only components with count>0, sort by count desc, limit to top 20 for display
error_agg_filtered = error_agg_full[error_agg_full['count'] > 0].sort_values('count', ascending=False)
error_summary = error_agg_filtered.head(20)

# Convert timestamp columns to string for compact display (ISO format)
if not error_summary.empty:
    error_summary['earliest_ts'] = error_summary['earliest_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    error_summary['latest_ts'] = error_summary['latest_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Up to 5 sample messages overall (not per-component)
sample_msgs = error_logs['message'].dropna().astype(str).unique()[:5]
error_sample_messages = pd.DataFrame({'sample_message': sample_msgs})

# Final displayed summaries (compact)
metric_summary, trace_summary, log_summary, error_summary, error_sample_messages

```
Out[1]:
```
summary = (
    "Global aggregation completed and saved in metric_agg_full, trace_agg_full, "
    "log_agg_full, and error_agg_full. Key findings from the top-20 summaries:\n"
    "- Metrics: Most metric groups have count = 31 (full series length). Tomcat04 appears "
    "frequently with many KPIs. Some metric maxima are very large (examples: a Tomcat04 "
    "MaxTime-related KPI max=230487.0, a Processing-related KPI mean≈9.25e8, RequestCount mean≈1.79e6).\n"
    "- Traces: Top trace groups also have count = 31. dockerA/dockerB -> MG* traces show "
    "mean durations ~0.17–0.22s and P95s ~0.6–1.1s; trace.row_count for dockerA nodes ~2100/minute.\n"
    "- Logs: log.row_count is high on apache01/apache02 (~880) and Tomcat nodes (~440). "
    "log.error_count is zero across the top components. gc_pause_max appears with values ~0.06.\n"
    "- Error logs: No error_log entries with count>0 were found and there are no sample messages.\n\n"
    "Next steps: I can (a) calculate thresholds (e.g., using the computed P95s) or (b) filter these "
    "aggregates to a specific incident time window—tell me which you prefer."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  count          mean           min           max           p95
55        MG01                             OSLinux-CPU_CPU_CPUWio     31  2.409000e-01  4.991500e-02  1.247600e+00  6.262730e-01
6         IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31  7.267742e+01  6.800000e+01  7.600000e+01  7.600000e+01
1119  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...     31  1.380000e+02  1.380000e+02  1.380000e+02  1.380000e+02
1129  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1136  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  4.193548e+00  0.000000e+00  1.000000e+01  9.000000e+00
1137  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...     31  5.000000e+02  5.000000e+02  5.000000e+02  5.000000e+02
1128  Tomcat04     Tomcat-Sessions_7441--_SESSIONRejectedSessions     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1131  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1130  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1132  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1133  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
847   Tomcat02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1134  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1135  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  1.000000e+02  1.000000e+02  1.000000e+02  1.000000e+02
1120  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...     31  2.304870e+05  2.304870e+05  2.304870e+05  2.304870e+05
1121  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...     31  9.254688e+08  9.216761e+08  9.291742e+08  9.288594e+08
1122  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...     31  1.787884e+06  1.781146e+06  1.794910e+06  1.794214e+06
1123  Tomcat04    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1124  Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1125  Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00,       cmdb_id                     trace_name  count         mean        min          max          p95
260  dockerB2        trace.to_MG01.row_count     31   163.903226   1.000000   268.000000   248.000000
259  dockerB2     trace.to_MG01.duration_p95     31     0.873803   0.006000     5.142000     3.329875
258  dockerB2    trace.to_MG01.duration_mean     31     0.172219   0.006000     1.663000     0.636816
219  dockerA2       trace.self.duration_mean     31     0.023627   0.010649     0.072190     0.049796
220  dockerA2        trace.self.duration_p95     31     0.010882   0.001000     0.106800     0.022200
221  dockerA2           trace.self.row_count     31  2131.000000  74.000000  3334.000000  3053.000000
222  dockerA2    trace.to_MG01.duration_mean     31     0.199482   0.007000     2.283632     0.730725
223  dockerA2     trace.to_MG01.duration_p95     31     0.872132   0.007000     5.320500     3.781425
224  dockerA2        trace.to_MG01.row_count     31    37.000000   1.000000    67.000000    60.500000
225  dockerA2    trace.to_MG02.duration_mean     31     0.216565   0.006333     1.651053     0.920796
226  dockerA2     trace.to_MG02.duration_p95     31     0.987165   0.007800     4.404000     3.434275
211  dockerA2   trace.from_MG01.duration_p95     31     1.066031   0.038300     2.636400     2.088650
212  dockerA2      trace.from_MG01.row_count     31   152.612903   5.000000   237.000000   211.500000
213  dockerA2  trace.from_MG02.duration_mean     31     0.182100   0.017400     0.635442     0.384552
214  dockerA2   trace.from_MG02.duration_p95     31     1.134842   0.021600     3.450000     1.899250
215  dockerA2      trace.from_MG02.row_count     31   152.516129   5.000000   238.000000   220.000000
203  dockerA1           trace.self.row_count     31  2119.741935  59.000000  3348.000000  3121.500000
204  dockerA1    trace.to_MG01.duration_mean     31     0.172168   0.006000     1.509667     0.655253
205  dockerA1     trace.to_MG01.duration_p95     31     0.784539   0.006900     3.335000     2.686225
206  dockerA1        trace.to_MG01.row_count     31    38.225806   3.000000    73.000000    66.000000,      cmdb_id          log_name  count        mean    min      max        p95
9   Tomcat02     log.row_count     31  440.387097  13.00   581.00   532.0000
7   Tomcat02   log.error_count     31    0.000000   0.00     0.00     0.0000
6   Tomcat01     log.row_count     31  439.000000   8.00   555.00   532.5000
4   Tomcat01   log.error_count     31    0.000000   0.00     0.00     0.0000
18  apache02   log.error_count     31    0.000000   0.00     0.00     0.0000
17  apache01     log.row_count     31  880.709677  15.00  1119.00  1041.5000
19  apache02     log.row_count     31  878.032258  20.00  1082.00  1030.5000
16  apache01   log.error_count     31    0.000000   0.00     0.00     0.0000
10  Tomcat03   log.error_count     31    0.000000   0.00     0.00     0.0000
15  Tomcat04     log.row_count     31  443.000000  13.00   555.00   542.0000
13  Tomcat04   log.error_count     31    0.000000   0.00     0.00     0.0000
12  Tomcat03     log.row_count     31  439.451613   6.00   524.00   512.0000
14  Tomcat04  log.gc_pause_max     24    0.063333   0.02     0.11     0.0900
8   Tomcat02  log.gc_pause_max     22    0.064091   0.02     0.10     0.0900
11  Tomcat03  log.gc_pause_max     22    0.064545   0.02     0.11     0.0995
5   Tomcat01  log.gc_pause_max     21    0.058095   0.02     0.08     0.0800
2    Mysql02   log.error_count      2    0.000000   0.00     0.00     0.0000
3    Mysql02     log.row_count      2    1.000000   1.00     1.00     1.0000
0    Mysql01   log.error_count      1    0.000000   0.00     0.00     0.0000
1    Mysql01     log.row_count      1    1.000000   1.00     1.00     1.0000, Empty DataFrame
Columns: [cmdb_id, count, earliest_ts, latest_ts]
Index: [], Empty DataFrame
Columns: [sample_message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# reuse existing variables: metric, metric_agg_full
tz = pytz.timezone('Asia/Shanghai')

# Define incident window in Asia/Shanghai (UTC+8) and convert to UTC epoch seconds
start_local = pd.Timestamp('2021-03-23 14:00:00').tz_localize(tz)
end_local = pd.Timestamp('2021-03-23 14:30:00').tz_localize(tz)
start_ts = int(start_local.tz_convert('UTC').timestamp())
end_ts = int(end_local.tz_convert('UTC').timestamp())

# Filter metric rows within the window (timestamps are epoch seconds)
metric_window = metric[(metric['timestamp'] >= start_ts) & (metric['timestamp'] <= end_ts)].copy()

# Merge with global P95 thresholds from metric_agg_full (these are global thresholds computed earlier)
# Ensure column names match: using cmdb_id and kpi_name
thresholds = metric_agg_full[['cmdb_id', 'kpi_name', 'p95']].copy()
metric_win_with_p95 = metric_window.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# Drop rows without a global p95 (no global aggregate) since thresholds must come from full series
metric_win_with_p95 = metric_win_with_p95.dropna(subset=['p95'])

# Mark anomalies: value >= global P95
metric_win_with_p95['is_anom'] = metric_win_with_p95['value'] >= metric_win_with_p95['p95']

# Save anomaly records for follow-up
metric_anoms_window = metric_win_with_p95[metric_win_with_p95['is_anom']].copy()

# Total number of anomalous metric points in window
total_anoms = int(metric_anoms_window.shape[0])

# Build compact summary grouped by cmdb_id and kpi_name
if total_anoms > 0:
    grp = metric_anoms_window.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
        anomaly_count_in_window=('is_anom','count'),
        earliest_anom_ts=('timestamp','min'),
        max_value_in_window=('value','max'),
        global_p95=('p95','first')
    )
    # convert earliest_anom_ts to Asia/Shanghai human-readable string
    grp['earliest_anomaly_timestamp'] = pd.to_datetime(grp['earliest_anom_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    # select and reorder columns as requested
    summary_df = grp[['cmdb_id','kpi_name','global_p95','anomaly_count_in_window','earliest_anomaly_timestamp','max_value_in_window']]
    # sort by anomaly_count desc then max_value desc
    summary_df = summary_df.sort_values(['anomaly_count_in_window','max_value_in_window'], ascending=[False, False]).head(20)
else:
    # empty summary with requested columns
    summary_df = pd.DataFrame(columns=['cmdb_id','kpi_name','global_p95','anomaly_count_in_window','earliest_anomaly_timestamp','max_value_in_window'])

# Display compact results: top-20 summary and total anomaly count
summary_df, total_anoms

```
Out[2]:
```
summary = (
    "Using the global P95 thresholds (from the full series) and filtering the metric data to "
    "2021-03-23 14:00–14:30 (UTC+8), we found 11,643 anomalous metric points in the window.\n\n"
    "Top findings (compact):\n"
    "- JVM memory (HeapMemoryMax) on IG01, IG02, MG01, MG02 exceeded the global P95 every minute "
    "(31 anomalies each); max observed = 2.040136e+09.\n"
    "- Tomcat request MaxTime metrics (various Tomcat nodes) exceeded P95 every minute (31 each); "
    "max values up to ~4.01656e+05.\n"
    "- Tomcat request ErrorCount metrics (Tomcat03/01/02/04 etc.) exceeded P95 every minute (31 each); "
    "example max values: Tomcat03=621, Tomcat01=560, Tomcat02=514, Tomcat04=138.\n"
    "- Tomcat MaxThreads metrics are at the 500 level and exceeded P95 every minute (31 each).\n"
    "- ServiceTest1 app.rr and app.sr exceeded P95 every minute (31 each) with max=100.\n\n"
    "Interpretation: The window shows persistent, system-wide anomalies concentrated in JVM memory "
    "and Tomcat request/error KPIs (repeated every minute across the 30-minute window). "
    "I saved the detailed anomalous records to metric_anoms_window for follow-up investigation."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(           cmdb_id                                           kpi_name    global_p95  anomaly_count_in_window earliest_anomaly_timestamp  max_value_in_window
0             IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09                       31    2021-03-23 14:00:00 CST         2.040136e+09
18            IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09                       31    2021-03-23 14:00:00 CST         2.040136e+09
36            MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09                       31    2021-03-23 14:00:00 CST         2.040136e+09
138           MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09                       31    2021-03-23 14:00:00 CST         2.040136e+09
3             IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.867203e+07                       31    2021-03-23 14:00:00 CST         5.867203e+07
21            IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.854114e+07                       31    2021-03-23 14:00:00 CST         5.854114e+07
984       Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  4.016560e+05                       31    2021-03-23 14:00:00 CST         4.016560e+05
748       Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  3.978080e+05                       31    2021-03-23 14:00:00 CST         3.978080e+05
864       Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  3.906190e+05                       31    2021-03-23 14:00:00 CST         3.906190e+05
1104      Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  2.304870e+05                       31    2021-03-23 14:00:00 CST         2.304870e+05
983       Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  6.210000e+02                       31    2021-03-23 14:00:00 CST         6.210000e+02
747       Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  5.600000e+02                       31    2021-03-23 14:00:00 CST         5.600000e+02
863       Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  5.140000e+02                       31    2021-03-23 14:00:00 CST         5.140000e+02
762       Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02                       31    2021-03-23 14:00:00 CST         5.000000e+02
881       Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02                       31    2021-03-23 14:00:00 CST         5.000000e+02
1001      Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02                       31    2021-03-23 14:00:00 CST         5.000000e+02
1121      Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02                       31    2021-03-23 14:00:00 CST         5.000000e+02
1103      Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  1.380000e+02                       31    2021-03-23 14:00:00 CST         1.380000e+02
604   ServiceTest1                                             app.rr  1.000000e+02                       31    2021-03-23 14:00:00 CST         1.000000e+02
605   ServiceTest1                                             app.sr  1.000000e+02                       31    2021-03-23 14:00:00 CST         1.000000e+02, 11643)```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

# reuse existing variables: trace, trace_agg_full, tz if available; otherwise reload tz
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure trace values are numeric
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')

# Define incident window in Asia/Shanghai (UTC+8) and convert to UTC epoch seconds
start_local = pd.Timestamp('2021-03-23 14:00:00').tz_localize(tz)
end_local = pd.Timestamp('2021-03-23 14:30:00').tz_localize(tz)
start_ts = int(start_local.tz_convert('UTC').timestamp())
end_ts = int(end_local.tz_convert('UTC').timestamp())

# Filter trace rows within the window (timestamps are epoch seconds)
trace_window = trace[(trace['timestamp'] >= start_ts) & (trace['timestamp'] <= end_ts)].copy()

# Prepare global p95 thresholds from trace_agg_full
thresholds_trace = trace_agg_full[['cmdb_id', 'trace_name', 'p95']].copy()

# Merge thresholds into windowed traces
trace_win_with_p95 = trace_window.merge(thresholds_trace, on=['cmdb_id', 'trace_name'], how='left')

# Drop rows without a global p95 (no global aggregate)
trace_win_with_p95 = trace_win_with_p95.dropna(subset=['p95'])

# Mark anomalies: value >= global P95
trace_win_with_p95['is_anom'] = trace_win_with_p95['value'] >= trace_win_with_p95['p95']

# Save detailed anomalous trace records for follow-up
trace_anoms_window = trace_win_with_p95[trace_win_with_p95['is_anom']].copy()

# Total number of anomalous trace points in window
total_trace_anoms = int(trace_anoms_window.shape[0])

# Build compact summary grouped by cmdb_id and trace_name
if total_trace_anoms > 0:
    grp = trace_anoms_window.groupby(['cmdb_id', 'trace_name'], as_index=False).agg(
        anomaly_count_in_window=('is_anom', 'count'),
        earliest_anom_ts=('timestamp', 'min'),
        max_value_in_window=('value', 'max'),
        global_p95=('p95', 'first')
    )
    # convert earliest_anom_ts to Asia/Shanghai human-readable string
    grp['earliest_anomaly_timestamp'] = pd.to_datetime(grp['earliest_anom_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    summary_df = grp[['cmdb_id', 'trace_name', 'global_p95', 'anomaly_count_in_window', 'earliest_anomaly_timestamp', 'max_value_in_window']]
    # sort and limit to top 20
    summary_df = summary_df.sort_values(['anomaly_count_in_window', 'max_value_in_window'], ascending=[False, False]).head(20)
else:
    summary_df = pd.DataFrame(columns=['cmdb_id','trace_name','global_p95','anomaly_count_in_window','earliest_anomaly_timestamp','max_value_in_window'])

# Display compact results: top-20 summary and total anomaly count
summary_df, total_trace_anoms

```
Out[3]:
```
summary = (
    "Trace anomaly summary for 2021-03-23 14:00–14:30 (UTC+8):\n"
    "- Total trace anomalies in the window: 512.\n"
    "- Highest per-group anomalies: Tomcat03.trace.self.duration_p95 had 8 anomalies (earliest 14:04) "
    "and Tomcat01.trace.self.duration_p95 had 7 anomalies (earliest 14:09). These indicate repeated high tail latencies on Tomcat services.\n"
    "- Several request/row_count spikes occurred between IG, Tomcat, and docker nodes (counts=2–3 per group). "
    "Examples: IG02→Tomcat03 and corresponding Tomcat03.from_IG02 row_count each had 3 anomalies (earliest ~14:05); "
    "IG01→Tomcat01 and Tomcat01.from_IG01 each had 3 anomalies (earliest ~14:16).\n"
    "- Docker nodes also show anomalies: dockerB2.trace.self.duration_p95 had 3 anomalies (max ~1.5436s); "
    "dockerB2 and dockerB1 had self.row_count spikes (maxes up to ~5,064 and ~4,468 respectively).\n"
    "- Overall pattern: concentrated tail latency anomalies on Tomcat self-duration metrics plus several elevated request/row_count spikes across IG↔Tomcat and docker hosts.\n"
    "Recommended next steps: investigate Tomcat tail latencies (GC, thread pool, backend calls) and correlate these times with the metric anomalies (CPU/memory) and logs saved earlier."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                   trace_name  global_p95  anomaly_count_in_window earliest_anomaly_timestamp  max_value_in_window
166  Tomcat03      trace.self.duration_p95       1.004                        8    2021-03-23 14:04:00 CST               1.0280
130  Tomcat01      trace.self.duration_p95       1.004                        7    2021-03-23 14:09:00 CST               1.0420
32       IG02  trace.to_Tomcat03.row_count     254.000                        3    2021-03-23 14:05:00 CST             283.0000
161  Tomcat03    trace.from_IG02.row_count     254.000                        3    2021-03-23 14:05:00 CST             283.0000
8        IG01  trace.to_Tomcat01.row_count     275.000                        3    2021-03-23 14:16:00 CST             276.0000
122  Tomcat01    trace.from_IG01.row_count     275.000                        3    2021-03-23 14:16:00 CST             276.0000
14       IG01  trace.to_Tomcat03.row_count     258.000                        3    2021-03-23 14:12:00 CST             270.0000
158  Tomcat03    trace.from_IG01.row_count     258.000                        3    2021-03-23 14:12:00 CST             270.0000
113      MG02  trace.to_dockerA2.row_count     220.000                        3    2021-03-23 14:07:00 CST             238.0000
215  dockerA2    trace.from_MG02.row_count     220.000                        3    2021-03-23 14:07:00 CST             238.0000
256  dockerB2      trace.self.duration_p95       1.010                        3    2021-03-23 14:09:00 CST               1.5436
257  dockerB2         trace.self.row_count    4103.000                        2    2021-03-23 14:19:00 CST            5064.0000
239  dockerB1         trace.self.row_count    3975.500                        2    2021-03-23 14:19:00 CST            4468.0000
149  Tomcat02         trace.self.row_count    3596.500                        2    2021-03-23 14:12:00 CST            4132.0000
131  Tomcat01         trace.self.row_count    3607.500                        2    2021-03-23 14:12:00 CST            3914.0000
185  Tomcat04         trace.self.row_count    3672.000                        2    2021-03-23 14:12:00 CST            3848.0000
167  Tomcat03         trace.self.row_count    3491.000                        2    2021-03-23 14:12:00 CST            3624.0000
203  dockerA1         trace.self.row_count    3121.500                        2    2021-03-23 14:07:00 CST            3348.0000
221  dockerA2         trace.self.row_count    3053.000                        2    2021-03-23 14:07:00 CST            3334.0000
23       IG02         trace.self.row_count    2093.000                        2    2021-03-23 14:12:00 CST            2236.0000, 512)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# reuse existing variables from kernel: metric_anoms_window, metric_agg_full
try:
    metric_anoms_window
except NameError:
    raise RuntimeError("metric_anoms_window not found in kernel. Run the anomaly detection step first.")

tz = pytz.timezone('Asia/Shanghai')

# Ensure necessary columns exist and types are correct
metric_anoms_window['value'] = pd.to_numeric(metric_anoms_window['value'], errors='coerce')
metric_anoms_window['p95'] = pd.to_numeric(metric_anoms_window['p95'], errors='coerce')
metric_anoms_window['timestamp'] = pd.to_numeric(metric_anoms_window['timestamp'], errors='coerce')

# Compute ratio value / global_p95 safely (avoid division by zero)
metric_anoms_window['ratio_over_p95'] = np.where(
    metric_anoms_window['p95'].fillna(0) != 0,
    metric_anoms_window['value'] / metric_anoms_window['p95'],
    np.nan
)

# Aggregate by cmdb_id
agg = metric_anoms_window.groupby('cmdb_id', as_index=False).agg(
    distinct_kpi_count = ('kpi_name', 'nunique'),
    total_anomaly_points = ('is_anom', 'count'),
    earliest_anom_ts = ('timestamp', 'min'),
    max_value_over_p95 = ('ratio_over_p95', 'max'),
    max_value_in_window = ('value', 'max')
)

# Convert earliest_anom_ts to Asia/Shanghai formatted string
if not agg.empty:
    agg['earliest_anomaly_timestamp'] = pd.to_datetime(agg['earliest_anom_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
else:
    agg['earliest_anomaly_timestamp'] = pd.Series(dtype=str)

# Reorder and round numeric columns for compact display
agg['max_value_over_p95'] = agg['max_value_over_p95'].round(4)
agg['max_value_in_window'] = agg['max_value_in_window'].round(6)

metric_faults_summary = agg[['cmdb_id','distinct_kpi_count','total_anomaly_points','earliest_anomaly_timestamp','max_value_over_p95','max_value_in_window']].copy()

# Sort and limit to top 20 as requested
metric_faults_summary = metric_faults_summary.sort_values(['total_anomaly_points','max_value_over_p95'], ascending=[False, False]).head(20)

# Display compact result and also the full summary shape for follow-up
metric_faults_summary, metric_faults_summary.shape

```
Out[4]:
```
Summary of metric anomaly aggregation by component (using global P95 thresholds):

- Top affected components by anomaly volume:
  - Mysql01: 1,960 anomaly points across 137 distinct KPIs. Earliest anomaly 2021-03-23 14:00 (CST). Highest ratio value/p95 = 1.875; max observed value ≈ 6.20e11.
  - Mysql02: 1,786 anomaly points across 137 KPIs. Earliest 14:00. Ratio max ≈ 1.0588; max value ≈ 2.75e12.
  - Tomcat02: 1,176 anomalies, 119 distinct KPIs; earliest 14:00; ratio max ≈ 1.6667; max value ≈ 4.19e9.
  - Tomcat03: 1,162 anomalies, 120 KPIs; earliest 14:00; ratio max = 12.0000 (largest relative breach); max value ≈ 4.19e9.
  - Tomcat04: 1,150 anomalies, 120 KPIs; earliest 14:00; ratio max ≈ 4.6975; max value ≈ 4.19e9.
  - Tomcat01: 1,108 anomalies, 117 KPIs; earliest 14:00; ratio max ≈ 3.3334; max value ≈ 4.19e9.
  - MG01 / MG02: 761 and 731 anomalies respectively, ~102–103 KPIs; earliest 14:00; MG02 has ratio max ≈ 7.13.
  - Redis02: 628 anomalies (76 KPIs), ratio max ≈ 2.1053, max value ≈ 6.13e6.
  - IG01 / IG02: ~130 and 127 anomalies, 18 KPIs each; earliest 14:00; ratio max ≈ 1.54 (IG01) and ≈1.21 (IG02).

- ServiceTest and apache nodes appear with moderate anomaly counts (~65–66 anomalies each across 4 KPIs), earliest 14:00, ratio max ~1.1–2.16 and max values in low hundreds.

- All components’ earliest anomalies occur at the start of the window (2021-03-23 14:00 CST), indicating anomalies were present from the beginning of the 14:00–14:30 window.

- Notable severity indicators:
  - Tomcat03 shows the largest relative breach (max value / global P95 = 12.0), suggesting some KPIs there far exceeded expected tail behavior.
  - Several components (Tomcats, Mysql, MGs) show very large absolute max values (billions to trillions), which may indicate measurement scale differences or extreme resource usage.

- The aggregated result is saved in the kernel variable metric_faults_summary for follow-up investigation (drill-down per KPI and timestamp).

Recommended next steps:
- Investigate Tomcat03 (high relative breach) and Tomcat cluster-wide issues (Tomcat01–04) for request/error/threads KPIs.
- Check Mysql01/02 for high-volume anomalies and inspect corresponding KPI types (e.g., I/O, queries, memory).
- Correlate these metric anomalies with trace_anoms_window and metric_anoms_window details, logs, and error traces to identify root causes (GC, thread exhaustion, backend latency, or I/O spikes).

The original code execution output of IPython Kernel is also provided below for reference:

(          cmdb_id  distinct_kpi_count  total_anomaly_points earliest_anomaly_timestamp  max_value_over_p95  max_value_in_window
4         Mysql01                 137                  1960    2021-03-23 14:00:00 CST              1.8750         6.199367e+11
5         Mysql02                 137                  1786    2021-03-23 14:00:00 CST              1.0588         2.753396e+12
20       Tomcat02                 119                  1176    2021-03-23 14:00:00 CST              1.6667         4.187619e+09
21       Tomcat03                 120                  1162    2021-03-23 14:00:00 CST             12.0000         4.187619e+09
22       Tomcat04                 120                  1150    2021-03-23 14:00:00 CST              4.6975         4.187619e+09
19       Tomcat01                 117                  1108    2021-03-23 14:00:00 CST              3.3334         4.187619e+09
2            MG01                 102                   761    2021-03-23 14:00:00 CST              2.2727         2.725841e+09
3            MG02                 103                   731    2021-03-23 14:00:00 CST              7.1317         2.728499e+09
7         Redis02                  76                   628    2021-03-23 14:00:00 CST              2.1053         6.130892e+06
0            IG01                  18                   130    2021-03-23 14:00:00 CST              1.5385         2.040136e+09
1            IG02                  18                   127    2021-03-23 14:00:00 CST              1.2091         2.040136e+09
6         Redis01                  11                    83    2021-03-23 14:00:00 CST              1.0000         1.000000e+02
23       apache01                  11                    81    2021-03-23 14:00:00 CST              1.6471         1.000000e+02
12   ServiceTest3                   4                    66    2021-03-23 14:00:00 CST              2.1573         4.210000e+02
9   ServiceTest10                   4                    66    2021-03-23 14:00:00 CST              1.5100         4.930000e+02
8    ServiceTest1                   4                    66    2021-03-23 14:00:00 CST              1.1749         3.380000e+02
10  ServiceTest11                   4                    66    2021-03-23 14:00:00 CST              1.1631         2.700000e+02
17   ServiceTest8                   4                    66    2021-03-23 14:00:00 CST              1.1274         3.760000e+02
14   ServiceTest5                   4                    66    2021-03-23 14:00:00 CST              1.1244         3.250000e+02
15   ServiceTest6                   4                    65    2021-03-23 14:00:00 CST              1.8724         3.600000e+02, (20, 6))```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse variables from kernel: metric_anoms_window, metric_agg_full
try:
    metric_anoms_window
    metric_agg_full
except NameError:
    raise RuntimeError("Required kernel variables metric_anoms_window or metric_agg_full not found.")

tz = pytz.timezone('Asia/Shanghai')

# Define incident window bounds in UTC epoch seconds (Asia/Shanghai -> UTC)
start_local = pd.Timestamp('2021-03-23 14:00:00').tz_localize(tz)
end_local = pd.Timestamp('2021-03-23 14:30:00').tz_localize(tz)
start_ts = int(start_local.tz_convert('UTC').timestamp())
end_ts = int(end_local.tz_convert('UTC').timestamp())

# Filter anomalous metric rows to Tomcat03 and the window (metric_anoms_window should already be windowed,
# but we re-filter to be safe)
tomcat03_df = metric_anoms_window[
    (metric_anoms_window['cmdb_id'] == 'Tomcat03') &
    (metric_anoms_window['timestamp'] >= start_ts) &
    (metric_anoms_window['timestamp'] <= end_ts)
].copy()

# Ensure p95 available from global aggregates (merge to be safe)
p95_map = metric_agg_full[['cmdb_id', 'kpi_name', 'p95']].drop_duplicates()
tomcat03_df = tomcat03_df.drop(columns=[c for c in ['p95'] if c in tomcat03_df.columns], errors='ignore') \
                         .merge(p95_map, on=['cmdb_id', 'kpi_name'], how='left')

# Ensure numeric and compute ratio
tomcat03_df['value'] = pd.to_numeric(tomcat03_df['value'], errors='coerce')
tomcat03_df['p95'] = pd.to_numeric(tomcat03_df['p95'], errors='coerce')
tomcat03_df['ratio_over_p95'] = np.where(tomcat03_df['p95'].fillna(0) != 0, tomcat03_df['value'] / tomcat03_df['p95'], np.nan)

# If no anomalous rows for Tomcat03, return empty result
if tomcat03_df.empty:
    tomcat03_faults = pd.DataFrame(columns=['kpi_name','run_id','run_start_ts','run_length_minutes','earliest_anomaly_timestamp','max_value_in_run','max_over_p95'])
    tomcat03_faults_filtered = tomcat03_faults.copy()
else:
    # Sort and identify consecutive-minute runs per kpi_name
    tomcat03_df = tomcat03_df.sort_values(['kpi_name', 'timestamp']).reset_index(drop=True)
    # Compute time diff per kpi_name
    tomcat03_df['ts_diff'] = tomcat03_df.groupby('kpi_name')['timestamp'].diff().fillna(0).astype(int)
    # A new run starts when ts_diff != 60 (i.e., not consecutive minute)
    tomcat03_df['new_run_flag'] = (tomcat03_df['ts_diff'] != 60).astype(int)
    # Cumulative sum per kpi to assign run ids
    tomcat03_df['run_id'] = tomcat03_df.groupby('kpi_name')['new_run_flag'].cumsum()

    # Aggregate runs (full run list)
    runs = (
        tomcat03_df
        .groupby(['kpi_name','run_id'], as_index=False)
        .agg(
            run_start_ts = ('timestamp','min'),
            run_length_minutes = ('timestamp','count'),
            earliest_anomaly_timestamp = ('timestamp','min'),
            max_value_in_run = ('value','max'),
            max_over_p95 = ('ratio_over_p95','max')
        )
    )
    # Convert earliest_anomaly_timestamp to Asia/Shanghai formatted string
    runs['earliest_anomaly_timestamp'] = pd.to_datetime(runs['earliest_anomaly_timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    # Keep run_start_timestamp as epoch seconds as requested
    runs['run_start_timestamp'] = runs['run_start_ts'].astype(int)

    # Save full run list in kernel variable tomcat03_faults (for follow-up)
    tomcat03_faults = runs.sort_values(['kpi_name','run_start_ts']).reset_index(drop=True)

    # Filter runs to length >= 3 minutes (fault runs)
    tomcat03_faults_filtered = tomcat03_faults[tomcat03_faults['run_length_minutes'] >= 3].copy()

    # Select and reorder columns as requested and sort by run_start_timestamp ascending
    tomcat03_faults_filtered = tomcat03_faults_filtered.sort_values('run_start_ts', ascending=True)
    tomcat03_faults_filtered = tomcat03_faults_filtered[[
        'kpi_name',
        'run_start_timestamp',
        'run_length_minutes',
        'earliest_anomaly_timestamp',
        'max_value_in_run',
        'max_over_p95'
    ]].reset_index(drop=True)
    # Round numeric columns for compactness
    tomcat03_faults_filtered['max_over_p95'] = tomcat03_faults_filtered['max_over_p95'].round(4)
    tomcat03_faults_filtered['max_value_in_run'] = tomcat03_faults_filtered['max_value_in_run'].round(6)

# Limit to top 20 rows for display
result = tomcat03_faults_filtered.head(20)

# Display compact results: the filtered runs (length>=3) and the full run list shape saved
result, tomcat03_faults.shape

```
Out[5]:
```
summary = (
    "Tomcat03 fault-run analysis (2021-03-23 14:00–14:30 UTC+8):\n"
    "- Many KPI fault runs for Tomcat03 were detected; the runs shown are long-lived and mostly start at 2021-03-23 14:00:00 (CST).\n"
    "- A large number of KPIs remain anomalous for the full window (run length = 31 minutes) — examples include:\n"
    "  * Tomcat-Requests \"MaxTime\" (max value in run = 401,656)\n"
    "  * Tomcat-Requests \"ErrorCount\" (max value in run = 621)\n    These indicate sustained high request latencies and elevated error counts throughout the 30-minute window.\n"
    "- Thread-related KPIs are saturated during the window (e.g., CurrentThreads=100, MaxThreads=500), consistent with thread pressure.\n"
    "- Several session and session-logging KPIs show anomalous values of 0 with undefined ratio over P95 (global P95 is zero), so ratio-based severity is not available for those.\n"
    "- Some OS/process KPIs (zabbix-related) show 29-minute runs, indicating persistent process-related anomalies as well.\n"
    "Implication and next steps: Tomcat03 experienced sustained, system-wide degradations across request latency, error rates, and thread/resource KPIs from the start of the window. Investigate Tomcat GC, thread pool saturation, backend dependency latency, and correlate these runs with metric_anoms_window, trace_anoms_window, and logs for root-cause analysis. The full run list is saved in kernel variable tomcat03_faults for follow-up."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                                             kpi_name  run_start_timestamp  run_length_minutes earliest_anomaly_timestamp  max_value_in_run  max_over_p95
0       OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies           1616479200                  29    2021-03-23 14:00:00 CST               1.0           1.0
1        OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT           1616479200                  31    2021-03-23 14:00:00 CST               0.0           NaN
2   Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...           1616479200                  31    2021-03-23 14:00:00 CST               0.0           NaN
3     Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter           1616479200                  31    2021-03-23 14:00:00 CST               0.0           NaN
4   Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...           1616479200                  31    2021-03-23 14:00:00 CST          401656.0           1.0
5   Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...           1616479200                  31    2021-03-23 14:00:00 CST             621.0           1.0
6          OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount           1616479200                  29    2021-03-23 14:00:00 CST               3.0           1.0
7   OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...           1616479200                  29    2021-03-23 14:00:00 CST               0.3           1.0
8   Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...           1616479200                  31    2021-03-23 14:00:00 CST               0.0           NaN
9         Tomcat-Sessions_7441--_SESSIONActiveCounter           1616479200                  31    2021-03-23 14:00:00 CST               0.0           NaN
10  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...           1616479200                  31    2021-03-23 14:00:00 CST               0.0           NaN
11  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...           1616479200                  31    2021-03-23 14:00:00 CST               0.0           NaN
12  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...           1616479200                  31    2021-03-23 14:00:00 CST               0.0           NaN
13  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...           1616479200                  31    2021-03-23 14:00:00 CST               0.0           NaN
14     Tomcat-Sessions_7441--_SESSIONRejectedSessions           1616479200                  31    2021-03-23 14:00:00 CST               0.0           NaN
15     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter           1616479200                  31    2021-03-23 14:00:00 CST               0.0           NaN
16  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...           1616479200                  31    2021-03-23 14:00:00 CST             100.0           1.0
17  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...           1616479200                  31    2021-03-23 14:00:00 CST             500.0           1.0
18  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...           1616479200                  31    2021-03-23 14:00:00 CST               0.0           NaN
19  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...           1616479200                  31    2021-03-23 14:00:00 CST               0.0           NaN, (543, 8))```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
import numpy as np

# Reuse existing kernel variables: log, log_agg_full, tz (UTC+8)
try:
    log
    log_agg_full
except NameError:
    raise RuntimeError("Required variables 'log' or 'log_agg_full' not found in kernel.")

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Define incident window in Asia/Shanghai (UTC+8) and convert to UTC epoch seconds
start_local = pd.Timestamp('2021-03-23 14:00:00').tz_localize(tz)
end_local = pd.Timestamp('2021-03-23 14:30:00').tz_localize(tz)
start_ts = int(start_local.tz_convert('UTC').timestamp())
end_ts = int(end_local.tz_convert('UTC').timestamp())

# Filter raw log rows for Tomcat03 within the window
tomcat03_logs_window = log[
    (log['cmdb_id'] == 'Tomcat03') &
    (log['timestamp'] >= start_ts) &
    (log['timestamp'] <= end_ts)
].copy()

# Prepare global p95 thresholds from precomputed log_agg_full (do NOT recompute from window)
thresholds_log = log_agg_full[['cmdb_id', 'log_name', 'p95']].drop_duplicates()

# Merge thresholds into windowed logs
tomcat03_logs_window = tomcat03_logs_window.merge(
    thresholds_log,
    on=['cmdb_id', 'log_name'],
    how='left',
    suffixes=('', '_global')
)

# Drop rows without a global p95 (no global aggregate available)
tomcat03_logs_window = tomcat03_logs_window.dropna(subset=['p95']).copy()

# Ensure numeric value
tomcat03_logs_window['value'] = pd.to_numeric(tomcat03_logs_window['value'], errors='coerce')
tomcat03_logs_window['p95'] = pd.to_numeric(tomcat03_logs_window['p95'], errors='coerce')

# Mark anomalies: value >= global p95
tomcat03_logs_window['is_anom'] = tomcat03_logs_window['value'] >= tomcat03_logs_window['p95']

# Save detailed anomalous log rows for follow-up
tomcat03_log_anoms_window = tomcat03_logs_window[tomcat03_logs_window['is_anom']].copy()

# Overall total number of log anomalies for Tomcat03 in the window
total_log_anoms = int(tomcat03_log_anoms_window.shape[0])

# Build compact summary grouped by log_name
if total_log_anoms > 0:
    grp = tomcat03_log_anoms_window.groupby('log_name', as_index=False).agg(
        anomaly_count_in_window = ('is_anom','count'),
        earliest_anom_ts = ('timestamp','min'),
        max_value_in_window = ('value','max'),
        global_p95 = ('p95','first')
    )
    # Convert earliest_anom_ts to Asia/Shanghai human-readable string
    grp['earliest_anomaly_timestamp'] = pd.to_datetime(grp['earliest_anom_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    # Reorder and round numeric columns for compact display
    summary_df = grp[['log_name','global_p95','anomaly_count_in_window','earliest_anomaly_timestamp','max_value_in_window']].copy()
    summary_df['global_p95'] = summary_df['global_p95'].round(6)
    summary_df['max_value_in_window'] = summary_df['max_value_in_window'].round(6)
    # Sort and limit to top 20
    summary_df = summary_df.sort_values(['anomaly_count_in_window','max_value_in_window'], ascending=[False, False]).head(20)
else:
    summary_df = pd.DataFrame(columns=['log_name','global_p95','anomaly_count_in_window','earliest_anomaly_timestamp','max_value_in_window'])

# Display compact results: summary table and total count
summary_df, total_log_anoms

```
Out[6]:
```
Summary for Tomcat03 logs in 2021-03-23 14:00–14:30 (UTC+8):

- Total log anomalous points found: 35 (saved to tomcat03_log_anoms_window).

- Per-log-name summary (compact):
  - log.error_count — global P95 = 0.0000 — anomaly_count = 31 — earliest = 2021-03-23 14:00:00 CST — max value = 0.00
    - Note: global P95 is zero, so entries with value 0 meet the >= P95 rule; ratio/severity is not informative here.
  - log.row_count — global P95 = 512.0000 — anomaly_count = 2 — earliest = 2021-03-23 14:13:00 CST — max value = 524.00
  - log.gc_pause_max — global P95 = 0.0995 — anomaly_count = 2 — earliest = 2021-03-23 14:14:00 CST — max value = 0.11

Interpretation / next steps:
- Many log.error_count rows were flagged because the P95 is zero; this requires checking whether P95=0 is meaningful or an artifact (and inspecting raw log content).
- The meaningful non-zero anomalies are small in count: two row_count spikes (~524) around 14:13 and two small GC pause peaks (~0.11s) around 14:14.
- You can now inspect tomcat03_log_anoms_window for the detailed anomalous rows and correlate those times with metric_anoms_window and trace_anoms_window for root-cause investigation.

The original code execution output of IPython Kernel is also provided below for reference:

(           log_name  global_p95  anomaly_count_in_window earliest_anomaly_timestamp  max_value_in_window
0   log.error_count      0.0000                       31    2021-03-23 14:00:00 CST                 0.00
2     log.row_count    512.0000                        2    2021-03-23 14:13:00 CST               524.00
1  log.gc_pause_max      0.0995                        2    2021-03-23 14:14:00 CST                 0.11, 35)```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz
import numpy as np

# Reuse timezone
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Define incident window in Asia/Shanghai and convert to UTC epoch seconds
start_local = pd.Timestamp('2021-03-23 14:00:00').tz_localize(tz)
end_local = pd.Timestamp('2021-03-23 14:30:00').tz_localize(tz)
start_ts = int(start_local.tz_convert('UTC').timestamp())
end_ts = int(end_local.tz_convert('UTC').timestamp())

# Prefer using precomputed trace_anoms_window if available
try:
    trace_anoms_window
    src_df = trace_anoms_window.copy()
    # Ensure it's limited to the window (safety)
    src_df = src_df[(src_df['timestamp'] >= start_ts) & (src_df['timestamp'] <= end_ts)]
except NameError:
    # Fallback: compute anomalies in the window using global thresholds trace_agg_full
    # Ensure trace and trace_agg_full exist
    if 'trace' not in globals() or 'trace_agg_full' not in globals():
        raise RuntimeError("trace or trace_agg_full not found in kernel; cannot proceed.")
    trace_window = trace[(trace['timestamp'] >= start_ts) & (trace['timestamp'] <= end_ts)].copy()
    thresholds_trace = trace_agg_full[['cmdb_id','trace_name','p95']].drop_duplicates()
    src_df = trace_window.merge(thresholds_trace, on=['cmdb_id','trace_name'], how='left').dropna(subset=['p95'])
    src_df['value'] = pd.to_numeric(src_df['value'], errors='coerce')
    src_df['is_anom'] = src_df['value'] >= src_df['p95']
    src_df = src_df[src_df['is_anom']].copy()

# Filter matches: cmdb_id == 'Tomcat03' OR trace_name contains specified substrings
pattern = r'to_Tomcat03|from_Tomcat03|to_Mysql|from_Mysql'
mask = (
    (src_df['cmdb_id'] == 'Tomcat03') |
    src_df['trace_name'].astype(str).str.contains(pattern, na=False)
)
tomcat03_trace_relations = src_df[mask].copy()

# Prepare the raw rows result (limit 20 sorted by timestamp ascending)
if not tomcat03_trace_relations.empty:
    tomcat03_trace_relations = tomcat03_trace_relations.sort_values('timestamp', ascending=True)
    tomcat03_trace_relations_head = tomcat03_trace_relations[['timestamp','cmdb_id','trace_name','value']].head(20).copy()
else:
    tomcat03_trace_relations_head = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value'])

# Aggregate earliest_anomaly_timestamp per distinct trace_name among matches (compact)
if not tomcat03_trace_relations.empty:
    earliest_per_trace = (
        tomcat03_trace_relations
        .groupby('trace_name', as_index=False)
        .agg(earliest_anom_ts=('timestamp','min'))
        .sort_values('earliest_anom_ts', ascending=True)
    )
    earliest_per_trace['earliest_anomaly_timestamp'] = pd.to_datetime(earliest_per_trace['earliest_anom_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    earliest_per_trace = earliest_per_trace[['trace_name','earliest_anomaly_timestamp']].head(20)
else:
    earliest_per_trace = pd.DataFrame(columns=['trace_name','earliest_anomaly_timestamp'])

# Output compact results: top-20 raw matched rows and earliest timestamps per trace_name
tomcat03_trace_relations_head, earliest_per_trace

```
Out[7]:
```
Summary of trace anomalies related to Tomcat03 (2021-03-23 14:00–14:30 UTC+8)

- I returned the top matched trace anomaly rows (sorted by timestamp ascending) — examples include:
  - 2021-03-23 14:04:00 CST — Tomcat03 — trace.self.duration_p95 = 1.004
  - 2021-03-23 14:05:00 CST — IG02 → trace.to_Tomcat03.row_count = 254 (and corresponding Tomcat03 trace.from_IG02.row_count = 254)
  - 2021-03-23 14:29:00 CST — multiple entries with higher durations: IG01/IG02 → trace.to_Tomcat03.duration_mean/p95, MG01/MG02 ← trace.from_Tomcat03.duration_mean/p95, and Tomcat03 self/from/to entries showing elevated duration_mean/p95 values (e.g., trace.self.duration_mean ≈ 0.182, trace.self.duration_p95 = 1.004; various duration_p95 values up to ~5–6s).

- Earliest-anomaly per trace_name (compact highlights, limited to 20):
  - trace.self.duration_p95 — earliest anomaly at 2021-03-23 14:04:00 CST
  - trace.to_Tomcat03.row_count & trace.from_IG02.row_count — earliest at 2021-03-23 14:05:00 CST
  - Several trace.duration_mean and duration_p95 entries (from/to IG01, IG02, MG01, MG02 and Tomcat03 self) first appear around 14:09:00–14:12:00 CST, with a cluster of stronger duration anomalies near 14:29:00 CST.

Interpretation / next steps
- There is an early tail-latency anomaly on Tomcat03 self-duration at ~14:04, then request/row_count spikes from IG02→Tomcat03 at ~14:05, and later larger propagation of elevated durations involving IG01/IG02 and MG01/MG02 around ~14:09–14:29. This suggests latency and load effects involving upstream (IG) and downstream (MG) interactions with Tomcat03 across the window.
- The full matched set is saved in tomcat03_trace_relations for detailed follow-up and correlation with metrics and logs.

The original code execution output of IPython Kernel is also provided below for reference:

(       timestamp   cmdb_id                         trace_name       value
1081  1616479440  Tomcat03            trace.self.duration_p95    1.004000
1190  1616479500      IG02        trace.to_Tomcat03.row_count  254.000000
1307  1616479500  Tomcat03          trace.from_IG02.row_count  254.000000
1309  1616479500  Tomcat03            trace.self.duration_p95    1.004000
2085  1616479740      IG01    trace.to_Tomcat03.duration_mean    1.305825
2086  1616479740      IG01     trace.to_Tomcat03.duration_p95    5.816200
2100  1616479740      IG02    trace.to_Tomcat03.duration_mean    1.256597
2101  1616479740      IG02     trace.to_Tomcat03.duration_p95    5.699750
2112  1616479740      MG01  trace.from_Tomcat03.duration_mean    1.098969
2113  1616479740      MG01   trace.from_Tomcat03.duration_p95    4.943200
2151  1616479740      MG02  trace.from_Tomcat03.duration_mean    0.891615
2152  1616479740      MG02   trace.from_Tomcat03.duration_p95    4.371200
2214  1616479740  Tomcat03      trace.from_IG01.duration_mean    1.305825
2215  1616479740  Tomcat03       trace.from_IG01.duration_p95    5.816200
2217  1616479740  Tomcat03      trace.from_IG02.duration_mean    1.256597
2218  1616479740  Tomcat03       trace.from_IG02.duration_p95    5.699750
2220  1616479740  Tomcat03           trace.self.duration_mean    0.182023
2221  1616479740  Tomcat03            trace.self.duration_p95    1.004000
2223  1616479740  Tomcat03        trace.to_MG01.duration_mean    1.098969
2224  1616479740  Tomcat03         trace.to_MG01.duration_p95    4.943200,                            trace_name earliest_anomaly_timestamp
13            trace.self.duration_p95    2021-03-23 14:04:00 CST
5           trace.from_IG02.row_count    2021-03-23 14:05:00 CST
23        trace.to_Tomcat03.row_count    2021-03-23 14:05:00 CST
3       trace.from_IG02.duration_mean    2021-03-23 14:09:00 CST
6   trace.from_Tomcat03.duration_mean    2021-03-23 14:09:00 CST
4        trace.from_IG02.duration_p95    2021-03-23 14:09:00 CST
7    trace.from_Tomcat03.duration_p95    2021-03-23 14:09:00 CST
0       trace.from_IG01.duration_mean    2021-03-23 14:09:00 CST
15        trace.to_MG01.duration_mean    2021-03-23 14:09:00 CST
12           trace.self.duration_mean    2021-03-23 14:09:00 CST
21    trace.to_Tomcat03.duration_mean    2021-03-23 14:09:00 CST
22     trace.to_Tomcat03.duration_p95    2021-03-23 14:09:00 CST
18        trace.to_MG02.duration_mean    2021-03-23 14:09:00 CST
19         trace.to_MG02.duration_p95    2021-03-23 14:09:00 CST
16         trace.to_MG01.duration_p95    2021-03-23 14:09:00 CST
1        trace.from_IG01.duration_p95    2021-03-23 14:09:00 CST
8       trace.from_Tomcat03.row_count    2021-03-23 14:12:00 CST
2           trace.from_IG01.row_count    2021-03-23 14:12:00 CST
20            trace.to_MG02.row_count    2021-03-23 14:12:00 CST
14               trace.self.row_count    2021-03-23 14:12:00 CST)```
```